# AgroguardAI-LLM Training

Fine-tune Mistral-7B-Instruct-v0.3 on the Agri-QA dataset using QLoRA.
Runs on a free T4 GPU (16GB VRAM) in ~45 minutes for 100 samples, 3 epochs.

**Before you start:** Mount Google Drive to save your trained adapter.


In [ ]:
# @title 1. Install dependencies
!pip install -qU transformers peft accelerate bitsandbytes trl datasets huggingface_hub
!pip install -qU xformers --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# @title 2. Mount Google Drive (saves adapter here)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/agroguardai'
!mkdir -p {DRIVE_PATH}

In [ ]:
# @title 3. Clone dataset from GitHub
!git clone https://github.com/agroguardaiaOS/agroguardai-llm.git /content/agroguardai-llm
!ls /content/agroguardai-llm/data/

In [ ]:
# @title 4. Preprocess the dataset
%cd /content/agroguardai-llm
!python src/preprocess.py \
    --data data/agri_qa.json \
    --output data/processed \
    --val-ratio 0.2 \
    --seed 42
!echo '---'
!wc -l data/processed/*.jsonl

In [ ]:
# @title 5. Run training (QLoRA on Mistral-7B)
# This takes ~45 min on a T4 GPU with 100 samples, 3 epochs
# Training runs in the background and you'll see loss decreasing every 10 steps

HF_TOKEN = ""  # @param {type:"string"}
# Leave HF_TOKEN empty to skip pushing to Hugging Face Hub

import os, yaml, json
from pathlib import Path

# Load config and optionally set hub token
cfg_path = '/content/agroguardai-llm/config/lora_config.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face Hub")
    cfg['output']['hub_model_id'] = 'agroguardaiaOS/agroguardai-llm-lora'
else:
    cfg['output']['hub_model_id'] = ''
    print("HF_TOKEN empty — adapter saved locally only")

# Write updated config
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f)

# Run training
!python src/train.py --config config/lora_config.yaml --data data/processed/

In [ ]:
# @title 6. Save adapter to Google Drive
!cp -r /content/agroguardai-llm/models/agroguardai-lora-adapter {DRIVE_PATH}/
!echo 'Adapter saved to Google Drive:'
!ls -lh {DRIVE_PATH}/agroguardai-lora-adapter/

In [ ]:
# @title 7. Test the model interactively (optional)
%cd /content/agroguardai-llm
!python src/inference.py \
    --base-model mistralai/Mistral-7B-Instruct-v0.3 \
    --adapter models/agroguardai-lora-adapter \
    --question 'My cassava leaf dey yellow and I see small white fly for under. Wetin I do?'